# OCR avec Tesseract

Lire le texte d’un document puis mesurer CER et WER. Test contrôlé typographique reproductible et option photo personnelle. Pas de compréhension sémantique garantie.

**Statut : code préparé et vérifié syntaxiquement, inférence non exécutée dans cet environnement.** Exécuter de haut en bas dans Colab ou Jupyter. Les sorties sont volontairement vides, aucun résultat inventé. Petits tests sans entraînement. Python 3.10–3.12 recommandé.

## 1. Installation
Dans Colab/Linux seulement, exécuter la cellule apt ci-dessous. Sous Windows installer Tesseract séparément et ajouter le binaire au PATH. Pour une photo réelle, renseigner IMAGE_PATH plus bas.

In [ ]:
%pip install "pytesseract>=0.3.13,<0.4" "pillow>=10,<13" "matplotlib>=3.8,<4"

In [ ]:
import shutil, subprocess, sys
if shutil.which('tesseract') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','tesseract-ocr','fonts-dejavu-core'],check=True)
    else:
        raise RuntimeError('Installer le binaire Tesseract et les données eng avant de continuer.')


In [ ]:
from pathlib import Path
import time, urllib.request
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
DATA=Path('data'); DATA.mkdir(exist_ok=True)
OUT=Path('outputs'); OUT.mkdir(exist_ok=True)
def download(url, name):
    path=DATA/name
    if not path.exists():
        with urllib.request.urlopen(url, timeout=60) as r:
            path.write_bytes(r.read())
    return path


## 2. Document de test ou image personnelle
L’entrée par défaut est un texte rendu, pas une photographie réelle. Pour votre photo, IMAGE_PATH pointe sur un fichier et REFERENCE doit contenir sa transcription exacte ; laisser REFERENCE=None pour désactiver les métriques.

In [ ]:
from PIL import ImageDraw, ImageFont, ImageFilter, ImageOps
import pytesseract
IMAGE_PATH=None
REFERENCE='INPT COMPUTER VISION\nOCR TEST 2026\nTOTAL 125.50'
if IMAGE_PATH:
    image=Image.open(IMAGE_PATH).convert('RGB')
    REFERENCE=None  # remplacer par la vérité terrain pour calculer les métriques
else:
    image=Image.new('RGB',(850,240),'white')
    try: font=ImageFont.truetype('DejaVuSans.ttf',38)
    except OSError: font=ImageFont.load_default(size=38)
    ImageDraw.Draw(image).multiline_text((30,28),REFERENCE,font=font,fill='black',spacing=20)
plt.figure(figsize=(12,3));plt.imshow(image);plt.axis('off');plt.show()
print(pytesseract.get_tesseract_version())


## 3. Reconnaissance
PSM 6 suppose un bloc de texte uniforme. Adapter ce réglage aux pages complexes. Le modèle anglais suffit au test ASCII ; installer fra pour le français.

In [ ]:
t0=time.perf_counter()
text=pytesseract.image_to_string(image,lang='eng',config='--psm 6').strip()
print(text); print(f'Temps : {time.perf_counter()-t0:.3f} s')
(OUT/'ocr.txt').write_text(text,encoding='utf-8')


## 4. CER, WER et flou
Distance de Levenshtein / longueur de la référence. Normalisation des espaces seulement, casse conservée. Le CER peut dépasser 100 % en présence de nombreuses insertions.

In [ ]:
def distance(a,b):
    row=list(range(len(b)+1))
    for i,x in enumerate(a,1):
        new=[i]
        for j,y in enumerate(b,1):new.append(min(new[-1]+1,row[j]+1,row[j-1]+(x!=y)))
        row=new
    return row[-1]
assert distance('chat','chats')==1
blur=image.filter(ImageFilter.GaussianBlur(3))
blur_text=pytesseract.image_to_string(blur,lang='eng',config='--psm 6').strip()
if REFERENCE:
    ref=' '.join(REFERENCE.split())
    for name,out in [('net',text),('flou',blur_text)]:
        out=' '.join(out.split())
        print(name,{'CER':distance(ref,out)/len(ref),'WER':distance(ref.split(),out.split())/len(ref.split())})
else: print('Métriques non calculées : pas de référence.')
plt.figure(figsize=(12,3));plt.imshow(blur);plt.axis('off');plt.show()


## Questions de compréhension
1. Détection du texte vs reconnaissance ?
2. CER vs WER ?
3. Pourquoi une référence exacte ?
4. Pourquoi tester sur des photos variées ?

## Bibliographie
- https://tesseract-ocr.github.io/tessdoc/
- https://tesseract-ocr.github.io/tessdoc/Command-Line-Usage.html
- https://pypi.org/project/pytesseract/